# First run: evaluation only, no training or publication
Pinned code, model and data; failure stops the notebook. Save outputs before session expiry.


In [ ]:
import subprocess, sys, os
from pathlib import Path
CODE_REVISION = "3b0afae8a511481e7593105dcb70aaf8be397a53"
BASE_REVISION = "93450be3f1ed40a930690d951ef3932687cc1892"
DATA_REVISION = "773832ea94643b630f63e8c2aa2002c634a7ade5"
repo = Path('/kaggle/working/OCR_engine')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/PiotrStyla/OCR_engine.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin',CODE_REVISION],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',CODE_REVISION],check=True)
os.chdir(repo)
sys.path.insert(0,str(repo))
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
subprocess.run([sys.executable,'-m','pip','install','transformers==4.57.6','peft==0.19.1','jiwer','pillow','accelerate'],check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), "GPU is required for the full evaluation; select a Kaggle GPU accelerator."
print(torch.cuda.get_device_name(0))
from huggingface_hub import snapshot_download
from training.protocol import pair_manifest
# All inputs are public; no HF key or secret printing is needed.
data_root = snapshot_download('PiotrSty/ocr-pl-lines',repo_type='dataset',revision=DATA_REVISION)
print('train:',len(pair_manifest(f'{data_root}/train')),'val:',len(pair_manifest(f'{data_root}/val')))


In [ ]:
output = '/kaggle/working/first-run-reevaluation'
subprocess.run([sys.executable,'-m','training.reevaluate_first_run',
    '--data',f'{data_root}/val','--base-revision',BASE_REVISION,
    '--output',output],check=True)
print(Path(output,'summary.json').read_text())
# Download the output directory. No training and no automatic Hub publication.
